**Installing Libraries and Running Data Audit**

In [ ]:
# Install Unsloth and dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

import torch
print("✅ Environment ready! GPU check:")
!nvidia-smi

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Environment ready! GPU check:
Thu Nov 12 07:25:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             30W /   70W |    2265MiB /  15360Mi

In [ ]:
import json
import csv
import os

# Updated path to point directly to train.jsonl in the root directory
train_path = "train.jsonl"
valid_categories = ["billing", "payment", "account_access", "kyc", "fraud", "other"]
valid_priorities = ["P1", "P2", "P3"]
valid_channels = ["email", "chat", "phone", "web"]

cleaning_log = []

if not os.path.exists(train_path):
    print(f"Error: {train_path} not found. Please make sure train.jsonl is uploaded!")
else:
    with open(train_path, "r") as f:
        for i, line in enumerate(f):
            row_id = f"line_{i+1}"
            try:
                data = json.loads(line)
                row_id = data.get("id", row_id)
                messages = data.get("messages", [])

                # Check chat structure
                if len(messages) < 3:
                    cleaning_log.append({"id": row_id, "action": "drop", "reason": "Missing chat turns (incomplete conversation)"})
                    continue

                assistant_msg = messages[2].get("content", "")
                parsed = json.loads(assistant_msg)

                # Validate categories and priorities against schema rules
                if parsed.get("category") not in valid_categories:
                    cleaning_log.append({"id": row_id, "action": "fix", "reason": f"Invalid category value: {parsed.get('category')}"})
                elif parsed.get("priority") not in valid_priorities:
                    cleaning_log.append({"id": row_id, "action": "fix", "reason": f"Invalid priority value: {parsed.get('priority')}"})
                elif parsed.get("channel") not in valid_channels:
                    cleaning_log.append({"id": row_id, "action": "fix", "reason": f"Invalid channel value: {parsed.get('channel')}"})

            except json.JSONDecodeError:
                cleaning_log.append({"id": row_id, "action": "drop", "reason": "JSON decode error / malformed structure"})
            except Exception as e:
                cleaning_log.append({"id": row_id, "action": "drop", "reason": str(e)})

    # Save cleaning log to CSV
    with open("cleaning_log.csv", "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=["id", "action", "reason"])
        writer.writeheader()
        writer.writerows(cleaning_log)

    print(f"✅ Audit complete! Logged {len(cleaning_log)} issues to 'cleaning_log.csv'.")

✅ Audit complete! Logged 1445 issues to 'cleaning_log.csv'.


**Baseline Evaluation**

In [ ]:
# Run the scorer on the dev set using your base model (we will generate baseline predictions first)
!python score.py --help

usage: score.py [-h] --gold GOLD --pred PRED [--json]

options:
  -h, --help   show this help message and exit
  --gold GOLD
  --pred PRED
  --json       print machine-readable JSON


In [ ]:
import json
import torch
from unsloth import FastLanguageModel
from tqdm import tqdm

# 1. Load model with 4096 sequence length
max_seq_length = 4096
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Set model to inference mode
FastLanguageModel.for_inference(model)

# 2. Load dev set inputs and run zero-shot inference
dev_path = "dev.jsonl"
predictions = []

print("Running zero-shot inference on dev set...")
with open(dev_path, "r") as f:
    for line in f:
        item = json.loads(line)
        item_id = item.get("id")
        messages = item.get("messages", [])

        # Extract user prompt
        user_prompt = messages[1]["content"] if len(messages) > 1 else messages[0]["content"]

        # Format prompt for Gemma-2 instruction model
        prompt = f"<bos><start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n"

        # Tokenize with explicit truncation
        inputs = tokenizer(
            [prompt],
            return_tensors="pt",
            truncation=True,
            max_length=max_seq_length
        ).to("cuda")

        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1, use_cache=True)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        # Extract response part
        response = decoded.split("model\n")[-1].strip()
        predictions.append({"id": item_id, "output": response})

# 3. Save baseline predictions
pred_path = "predictions_dev_baseline.jsonl"
with open(pred_path, "w") as out:
    for p in predictions:
        out.write(json.dumps(p) + "\n")

print(f"✅ Baseline predictions saved to {pred_path}!")

==((====))==  Unsloth 2026.11.4: Fast Gemma2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running zero-shot inference on dev set...


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

✅ Baseline predictions saved to predictions_dev_baseline.jsonl!


**Running the Scorer on the Baseline Predictions**

In [ ]:
!python score.py --gold dev.jsonl --pred predictions_dev_baseline.jsonl

group                            n   valid   exact  fields
all                            200    0.0%    0.0%    0.0%
category=account_access         19    0.0%    0.0%    0.0%
category=fraud                  15    0.0%    0.0%    0.0%
category=kyc                    24    0.0%    0.0%    0.0%
category=offers                 18    0.0%    0.0%    0.0%
category=other                  16    0.0%    0.0%    0.0%
category=payment_failed         53    0.0%    0.0%    0.0%
category=refund                 55    0.0%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%


**Fine tuning the model (QLoRA)**

In [ ]:
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# 1. Add LoRA adapters (keeps trainable parameters well under limits)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ LoRA adapters successfully added!")

# 2. Format and load dataset for training
# (Using your train.jsonl file)
dataset = load_dataset("json", data_files="train.jsonl", split="train")

def formatting_prompts_func(examples):
    messages_list = examples["messages"]
    texts = []
    for messages in messages_list:
        if len(messages) >= 3:
            user_msg = messages[1]["content"] if len(messages) > 1 else messages[0]["content"]
            assistant_msg = messages[2]["content"]

            # Gemma-2 format
            text = f"<bos><start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{assistant_msg}<end_of_turn><eos>"
            texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# 3. Configure Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    dataset_num_proc = 2,
    packing = False, # Can speed up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Adjust steps as needed for a quick fine-tune
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

print("🚀 Starting fine-tuning training loop...")
trainer_stats = trainer.train()
print("✅ Training complete!")

Unsloth 2026.11.4 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Unsloth: compiling each transformer block as one region (Gemma2DecoderLayer).
✅ LoRA adapters successfully added!


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1445 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1445 [00:00<?, ? examples/s]

... (2 outputs left out)

Step,Training Loss
1,3.986994
2,3.718952
3,3.798284
4,3.454648
5,3.022171
6,2.550627
7,2.227707
8,1.025424
9,1.681852
10,1.526007


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


✅ Training complete!


**Save the LoRA Adapters & Generate Test Predictions**

In [13]:
import json
import torch
from unsloth import FastLanguageModel
from tqdm import tqdm

test_path = "test_inputs.jsonl"
print(f"🚀 Running fast batched inference on {test_path}...")

# Load test items
items = []
with open(test_path, "r") as f:
    for line in f:
        items.append(json.loads(line))

test_predictions = []
batch_size = 8  # Process in small batches for speed & safety

for i in tqdm(range(0, len(items), batch_size)):
    batch = items[i:i + batch_size]
    prompts = []

    for item in batch:
        messages = item.get("messages", [])
        user_prompt = messages[1]["content"] if len(messages) > 1 else messages[0]["content"]
        prompt = f"<bos><start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n"
        prompts.append(prompt)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for item, dec in zip(batch, decoded):
        item_id = item.get("id")
        response = dec.split("model\n")[-1].strip()
        test_predictions.append({"id": item_id, "output": response})

# Save final test predictions file
output_test_path = "predictions_test.jsonl"
with open(output_test_path, "w") as out:
    for p in test_predictions:
        out.write(json.dumps(p) + "\n")

print(f"✅ Test predictions successfully saved to {output_test_path}!")

🚀 Running fast batched inference on test_inputs.jsonl...


100%|██████████| 50/50 [10:21<00:00, 12.42s/it]

✅ Test predictions successfully saved to predictions_test.jsonl!
